# NextQA Evaluation Notebook

Colab + Google Drive 기준으로 `eval.nextqa`와 `eval.nextqa_val`을 바로 실행하기 위한 노트북입니다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


아래 경로를 현재 드라이브 구조에 맞게 확인하세요.

In [2]:
REPO_DIR = '/content/drive/MyDrive/KMK/GUARD_EXP_REPO'
NEXTQA_ROOT = '/content/drive/MyDrive/NExTVideo/nextqa'

In [3]:
%cd {REPO_DIR}
!pwd
!ls

/content/drive/MyDrive/KMK/GUARD_EXP_REPO
/content/drive/MyDrive/KMK/GUARD_EXP_REPO
config	model		       requirements.launch.txt	Untitled0.ipynb
data	nextqa_terminal.ipynb  requirements.txt		utils
doc	outputs		       setting.sh
eval	readme.md	       terminal.ipynb


NextQA 데이터 경로가 맞는지 간단히 확인합니다.

In [4]:
!ls {NEXTQA_ROOT}
!ls {NEXTQA_ROOT}/videos | head

glove_embed.npy  map_vid_vidorID.json  train.csv  val.csv  videos  vocab.pkl
0000
0001
0002
0003
0004
0005
0006
0007
0008
0009


필요하면 아래 셀에서 의존성을 설치하세요.

In [5]:
# !pip install -r requirements.txt


현재 `nextqa` 설정을 확인합니다.

In [6]:
!sed -n '1,120p' config/eval.yaml
!echo '-----'
!sed -n '1,120p' config/val.yaml

defaults:
  - override hydra/job_logging: disabled
  - override hydra/hydra_logging: disabled

hydra:
  job:
    chdir: false
  output_subdir: null
  run:
    dir: .

experiment: "base" # 사용할 모델 config 파일 이름

egoschema:
  dataset_root: "../../EgoSchema" # 데이터 셋 루트 경로, question파일이랑 map파일 값이 null이면 자동으로 데이터 셋 루트에서 탐색함
  questions_file: "../../EgoSchema/questions.json"
  uid_map_file: "../../EgoSchema/EgoSchema/uid_to_url.json"
  videos_dir: "../../EgoSchema/EgoSchema/Egochema_videos"
  output_dir: "./eval/result"
  output_file: "./eval/result/ego_pred.jsonl"
  resume: false # 이전에 연산해둔거 있으면 이어서 실행
  progress_interval: 10 # tqdm 업데이트 간격
  start_index: 0 # 시작 샘플 인덱스, 즉 limit이 300이면 0~299의 샘플을 사용
  limit: null # 테스트할 샘플 수, null로 설정되면 전체 평가

nextqa:
  dataset_root: "/content/drive/MyDrive/NExTVideo/nextqa" # Colab + Google Drive default root
  questions_file: "/content/drive/MyDrive/NExTVideo/nextqa/val.csv"
  videos_dir: "/content/drive/MyDrive/NExTVideo/nextqa/videos"
  output_dir: "./eval/

In [7]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes


Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)


NextQA 평가를 실행합니다. `experiment` 값을 `base`, `DDPS`, `trips`, `trips_budget` 등으로 바꿔서 사용할 수 있습니다.

`trips` 실험을 바로 실행하려면 아래 셀을 사용하세요.

In [8]:
# # 아래와 같이 .yaml 수정 없이 바로 cli로 옵션 선택 가능
# LIMIT = "null"

# !python -m eval.nextqa experiment=trips nextqa.dataset_root={NEXTQA_ROOT} nextqa.questions_file={NEXTQA_ROOT}/val.csv nextqa.videos_dir={NEXTQA_ROOT}/videos nextqa.limit={LIMIT}

`trips_budget` 실험을 바로 실행하려면 아래 셀을 사용하세요.

In [9]:
# # 아래와 같이 .yaml 수정 없이 바로 cli로 옵션 선택 가능
# LIMIT = "null"

# !python -m eval.nextqa experiment=trips_budget nextqa.dataset_root={NEXTQA_ROOT} nextqa.questions_file={NEXTQA_ROOT}/val.csv nextqa.videos_dir={NEXTQA_ROOT}/videos nextqa.limit={LIMIT}

In [13]:
import sys, site, glob, os
from pathlib import Path

print("python:", sys.version)
print("site:", site.getsitepackages())

matches = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/**/lib/libnvJitLink.so*", recursive=True)
print(matches[:20])


python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
site: ['/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/lib/python3.12/dist-packages']
['/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib/libnvJitLink.so.12', '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13']


In [14]:
import bitsandbytes as bnb
print(bnb.__version__)


ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


0.49.2


In [15]:
import glob

matches = glob.glob(
    "/usr/local/lib/python3.12/dist-packages/nvidia/**/libnvJitLink.so*",
    recursive=True,
)
print(matches)


['/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib/libnvJitLink.so.12', '/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib/libnvJitLink.so.13']


In [16]:
import os, site, glob

lib_dirs = []
for root in site.getsitepackages():
    lib_dirs += glob.glob(f"{root}/nvidia/**/lib", recursive=True)

lib_dirs = sorted(set(lib_dirs))
print("\n".join(lib_dirs))

os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("LD_LIBRARY_PATH =", os.environ["LD_LIBRARY_PATH"])


/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cublas/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cccl/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_cupti/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvcc/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_nvrtc/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cuda_runtime/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cudnn/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cufft/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cufile/lib
/usr/local/lib/python3.12/dist-packages/nvidia/curand/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cusolver/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cusparse/lib
/usr/local/lib/python3.12/dist-packages/nvidia/cusparselt/lib
/usr/local/lib/python3.12/dist-packages/nvidia/nccl/lib
/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib
/usr/local/lib/python3.12/dist-packages/nvidia/

In [17]:
import subprocess, os

subprocess.run(
    ["python", "-c", "import bitsandbytes as bnb; print(bnb.__version__)"],
    env=os.environ.copy(),
    check=True,
)


CompletedProcess(args=['python', '-c', 'import bitsandbytes as bnb; print(bnb.__version__)'], returncode=0)

In [ ]:
EXPERIMENT = 'prototype_vtcp'
LIMIT = "null"

!python -m eval.nextqa experiment={EXPERIMENT} nextqa.dataset_root={NEXTQA_ROOT} nextqa.questions_file={NEXTQA_ROOT}/val.csv nextqa.videos_dir={NEXTQA_ROOT}/videos nextqa.limit={LIMIT}

=== NextQA Evaluation Setup ===
Experiment   : /content/drive/MyDrive/KMK/GUARD_EXP_REPO/config/prototype_vtcp.yaml
Dataset Root : /content/drive/.shortcut-targets-by-id/1yn_sJZbnaTlKybCiQd-jm-8aqmvpM3YJ/NExTVideo/nextqa
Questions    : /content/drive/.shortcut-targets-by-id/1yn_sJZbnaTlKybCiQd-jm-8aqmvpM3YJ/NExTVideo/nextqa/val.csv
Videos Dir   : /content/drive/.shortcut-targets-by-id/1yn_sJZbnaTlKybCiQd-jm-8aqmvpM3YJ/NExTVideo/nextqa/videos
Prompt File  : /content/drive/MyDrive/KMK/GUARD_EXP_REPO/model/base/prompt.txt
Output File  : /content/drive/MyDrive/KMK/GUARD_EXP_REPO/eval/result/nextqa_pred.jsonl
Samples      : 4996
Pending      : 4996

Loading weights:   0% 0/750 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 750/750 [00:02<00:00, 253.92i

생성된 예측 파일 일부를 확인합니다.

In [19]:
!ls eval/result
!head -n 3 eval/result/nextqa_pred.jsonl

nextqa_pred.jsonl
{"q_uid": "4010069381:6", "video_id": "4010069381", "video_path": "/content/drive/.shortcut-targets-by-id/1yn_sJZbnaTlKybCiQd-jm-8aqmvpM3YJ/NExTVideo/nextqa/videos/1106/4010069381.mp4", "query_text": "how do the two man play the instrument", "question": "how do the two man play the instrument", "options": ["roll the handle", "tap their feet", "strum the string", "hit with sticks", "pat with hand"], "ground_truth": 0, "prediction": 2, "prediction_text": "strum the string", "correct": false, "parse_method": "index_regex", "response": "2", "question_type": "CH", "raw_item": {"video": "4010069381", "frame_count": "369", "width": "640", "height": "480", "question": "how do the two man play the instrument", "answer": "0", "qid": "6", "type": "CH", "a0": "roll the handle", "a1": "tap their feet", "a2": "strum the string", "a3": "hit with sticks", "a4": "pat with hand"}, "dynamic_query_file": "/tmp/nextqa_query_bhb7gam9/query.txt"}
{"q_uid": "4882821564:1", "video_id": "48828

예측 결과와 `val.csv`를 비교해 accuracy를 계산합니다.

In [20]:
!python -m eval.nextqa_val nextqa_val.dataset_root={NEXTQA_ROOT} nextqa_val.questions_file={NEXTQA_ROOT}/val.csv

Loaded predictions : 19
Labeled samples    : 4996
Matched samples    : 19
Correct samples    : 13
Accuracy           : 0.6842
Missing preds      : 4977
Extra preds        : 0
